In [1]:

import numpy as np
from scipy.integrate import odeint
from pysindy.feature_library import PolynomialLibrary
import matplotlib.pyplot as plt
import pysindy as ps
from scipy.optimize import fsolve
from pysindy.differentiation import FiniteDifference
import pandas
from mpl_toolkits import mplot3d
from scipy.optimize import curve_fit
from pysindy.feature_library import CustomLibrary
import math
import pandas as pd
import torch
import torch.optim as optim

In [2]:
#!pip install pysindy # I needed this to use colab. You do not need to run this

In [3]:
#!pip install scipy==1.10.1 # I needed this to use colab. You do not need to run this

In [4]:
#!pip install --force-reinstall numpy==1.23.5 #needed for colab. You do not need to run

In [5]:
#Original Lotka Volterra Model
def LV_model(z,t,pars):
    x = z[0]
    y = z[1]
    #The model
    alpha,beta,delta,gamma = pars
    dxdt = alpha*x-beta*x*y
    dydt = delta*x*y-gamma*y
    dzdt = [dxdt,dydt]
    return dzdt

In [6]:
#model of polynomial basis functions up to order 2
def SINDY_model(z,t,pars):
    #function dydt=toy_model(y,t,pars)
    x = z[0]
    y = z[1]
    #The model
    a0,a1,a2,a3,a4,a5,b0,b1,b2,b3,b4,b5 = pars
    dxdt = a0+a1*x+a2*y+a3*x*x+a4*x*y+a5*y*y
    dydt = b0+b1*x+b2*y+b3*x*x+b4*x*y+b5*y*y
    dzdt = [dxdt,dydt]
    return dzdt

In [7]:
#initialize parameters
pars = {}
#These parameters stay the same for all trajectories
alpha=0.5
gamma=0.5


num_init_cond = 4
beta_values=[0.5,0.75,1]
delta_values=[0.5,0.75,1]

#choose initial values from a uniform distribution
#x_init_values = np.random.uniform(1,3,num_init_cond)
#y_init_values = np.random.uniform(1,3,num_init_cond)
t = np.linspace(0, 10, 100)

#Once chosen, fix the initial values to be the same for all iterations
x_init_values=[2.90311835,2.08080383,1.92179248,2.54818293]
y_init_values=[2.8146938,2.05761677,1.45111475,2.45620932]

print(x_init_values)
print(y_init_values)

[2.90311835, 2.08080383, 1.92179248, 2.54818293]
[2.8146938, 2.05761677, 1.45111475, 2.45620932]


In [8]:
#generate an initial guess for the common components of beta and delta
#generate trajectories for each combination of beta, delta, and initial conditions, recover beta and delta using SINDY
#take the average of the recovered betas and deltas, this is our beta_c and delta_c, respectively
MSE_values=[]
RSE_values=[]

recovered_beta=np.array([]) #array to store all beta values recovered by SinDY
recovered_delta=np.array([]) #array to store all delta values recovered by SinDY
for beta_value in beta_values: #iterate thru each combination of beta,delta,initial condition
    for delta_value in delta_values:
        for i in range(num_init_cond):
            pars=[alpha,beta_value,delta_value,gamma] #set parameters as the true values
            x_values=[]
            y_values=[]
            T=[]
            x0=x_init_values[i]
            y0=y_init_values[i]
            z=odeint(LV_model,np.array([x0,y0]),t,args=(pars,)) #generate trajectory
            x_values=np.append(x_values,z[:,0])
            y_values=np.append(y_values,z[:,1])
            T=np.append(T,t)
            W = np.stack((x_values, y_values), axis=-1)
            model = ps.SINDy(feature_names=["x", "y"]) #use SINDY to fit a model to the trajectory
            model.fit(W, t=T)
            #model.print()

            S=model.score(W, t=T)
            C= model.coefficients()
            #print(C)

            beta_r=-C[0,4] #coefficient that corresponds to beta
            delta_r=C[1,4] #coefficient that corresponds to delta
            #print(C[0,4])
            x_values_sindy=[]
            y_values_sindy=[]
            T=[]

            recovered_pars=[alpha,beta_r,gamma,delta_r] #set the parameters to the recovered values
            z_hat=odeint(LV_model,np.array([x0,y0]),t,args=(recovered_pars,)) #generate estimated trajectory with the parameters recovered from SINDy
            x_values_sindy=np.append(x_values_sindy,z[:,0])
            y_values_sindy=np.append(y_values_sindy,z[:,1])
            T=np.append(T,t)

            recovered_beta=np.append(recovered_beta, C[0,4]) #beta is the coefficient of xy in the equation for x'
            recovered_delta=np.append(recovered_delta, C[1,4]) #delta is the coefficient of xy in the equation for y'
            MSE = np.square(np.subtract(z,z_hat)).mean() #compare the original trajectory to the one generated using SINDy, this is the MSE generated by SINDy
            RSE= MSE/np.linalg.norm(z) #Rel squared error
            MSE_values=np.append(MSE_values,MSE)
            RSE_values=np.append(RSE_values,RSE)


print("The average MSE is", np.average(MSE_values)) #We store the MSE for each trajectory and then take the average
print("The average RSE is", np.average(RSE_values)) #do the same for the RSE

beta_c=-np.average(recovered_beta) #Our initial guess for the common components is the average of the parameter values recovered by sindy
delta_c=np.average(recovered_delta)
plt.show()
print("beta c is", beta_c)
print("delta c is", delta_c)

The average MSE is 1.237247947700278
The average RSE is 0.06927300912300212
beta c is 0.6993538843651249
delta c is 0.7415120642347586


In [9]:
#the initial guess for the parameters
#we will update the parameter values using Adam
#We initialize the common components as the average beta/delta recovered by SINDy
#Initialize the personalize components as zero

init_guess_common=[beta_c,delta_c] #initialize the common component

#As there is a personalized component for each trajectory, the number of personalized components is (num trajectories*num parameters being personalized)
# There are len(beta_values)*len(delta_values)*num_init_cond total trajectories
#there are len(init_guess_common) parameters being personalized (beta and delta)
#Thus there are len(beta_values)*len(delta_values)*num_init_cond*len(init_guess_common) total personalized components
init_guess_pers=np.zeros(len(init_guess_common)*len(beta_values)*len(delta_values)*num_init_cond)

init_guess=np.concatenate((init_guess_common,init_guess_pers)) #put both common and pers comps into a single vector
init_guess=np.c_[init_guess] #make the correct shape

print(init_guess)
print(init_guess.shape)

init_guess=torch.tensor(init_guess, requires_grad=True) #convert to a tensor to use PyTorch

[[0.69935388]
 [0.74151206]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.        ]
 [0.  

In [10]:
#Loss function for the training
#Takes in a tensor of parameter values as input
#Returns sum_{i=1}^{n} ||X_dot-Theta(X)(Eps_c+Eps_p)||_2^2+||Eps_c+Eps_p_i||_1+||Eps_p_i||_2 (the loss function)
def loss(init_guess):
  sum=0 #initialize sum
  count=0
  for i in range(len(beta_values)): #loop through each beta,delta, and init_cond to generate a trajectory
    for j in range(len(delta_values)):
      for k in range(num_init_cond):
            x0=x_init_values[k]
            y0=y_init_values[k]
            params=[alpha,beta_values[i],delta_values[j],gamma] #generate trajectories with true params
            results=odeint(LV_model,[x0,y0],t,args=(params,))
            results= np.array(results)
            W=np.c_[results]

            #separate the data into x and y values
            x_values=W[:,0] #x-values
            y_values=W[:,1] #y-values
            fd=FiniteDifference()
            W_prime=fd._differentiate(W,t) #Take the derivative using fd


            #convert everything into tensors
            x_values_tensor = torch.tensor(x_values, dtype=init_guess.dtype)
            y_values_tensor = torch.tensor(y_values, dtype=init_guess.dtype)
            W_prime_tensor = torch.tensor(W_prime, dtype=init_guess.dtype)


            #Estimate the derivatives
            #W_prime is the derivative using finite difference methods
            #W_prime_hat is the estimated derivative (the right hand side of ODE, with beta=beta_c+beta_p_(count) and delta=delta_c+delta_p_(count)

            W_prime=W_prime_tensor

            #Estimate the derivative using the RHS of the Lotka_Volterra eqns
            X_hat=alpha*x_values_tensor-(init_guess[0]+init_guess[count+2])*x_values_tensor*y_values_tensor #estimate the RHS of x
            Y_hat=(init_guess[1]+init_guess[len(beta_values)*len(delta_values)*num_init_cond+count+2])*x_values_tensor*y_values_tensor-gamma*y_values_tensor #es
            W_prime_hat=torch.stack((X_hat,Y_hat),dim=1)

            #i.e.
            #X_hat=alpha*x-(beta_c+beta_p_(count))*xy
            #Y_hat=(delta_c+delta_c_(count))*xy-gamma*y


            #Note: I iterate through the pers components using a count variable.
            #count is increased by 1 for each trajectory
            #beta_c is at init_guess[0], delta_c is at init_guess[1], below them are all of the beta_p's and then below them, the delta_p's
            #beta_p_(count) is at init_guess[count+2] #the beta_p's start at index 2
            #delta_p_(count) is at init_guess[num_trajectories+count+2]

            Lambda_1=0.001 #Can adjust these later on
            Lambda_2=0.001 #Can adjust later on

            #Take the loss function for each trajectory and then sum them up
            sum+=torch.linalg.norm(W_prime-W_prime_hat)**2 + Lambda_1*torch.linalg.norm(torch.tensor([init_guess[0]+init_guess[count+2],init_guess[1]+init_guess[len(beta_values)*len(delta_values)*num_init_cond+count+2]]), ord=1) + Lambda_2*(torch.linalg.norm(torch.tensor([init_guess[count+2],init_guess[len(beta_values)*len(delta_values)*num_init_cond+count+2]])))**2
            count+=1
  return sum

In [11]:
print(loss(init_guess))

tensor(567.1759, dtype=torch.float64, grad_fn=<AddBackward0>)


In [12]:
#Run the Adam algorithm
init_guess=torch.tensor(init_guess, requires_grad=True)
optimizer=optim.Adam([init_guess],lr=0.001) #creates an Adam Optimizer object, lr is learning rate, use the default value from PyTorch
prev=0 #represent the loss value of the previous iteration
step=0 #the iteration number
loss_value=100 #use to represent the current loss value; I initialized as a random value just so loss_value and prev wouldn't start out the same

while(abs(loss_value-prev)>0.0001): #until convergence (algorithm terminates when current and previous loss function are within 0.0001 of each other)
  prev=loss_value #set prev=the loss value of the previous iteration
  optimizer.zero_grad() #reset the gradient
  loss_value=loss(init_guess) #take loss value of current iteration
  loss_value.backward() #take the gradient
  optimizer.step() #update parameters using Adam, step forward
  print("This is step", step, "and the loss is", loss_value.item())
  step+=1

This is step 0 and the loss is 567.1758862257021
This is step 1 and the loss is 561.6830468578039
This is step 2 and the loss is 556.2590803646317
This is step 3 and the loss is 550.9047835854625
This is step 4 and the loss is 545.6202726167143
This is step 5 and the loss is 540.4046392248372
This is step 6 and the loss is 535.2556993171046
This is step 7 and the loss is 530.1701872921725
This is step 8 and the loss is 525.1445048162619
This is step 9 and the loss is 520.1755127471295
This is step 10 and the loss is 515.2608633445886
This is step 11 and the loss is 510.39893604754366
This is step 12 and the loss is 505.58864183598894
This is step 13 and the loss is 500.8292364056427
This is step 14 and the loss is 496.1201765707906
This is step 15 and the loss is 491.4610186961584
This is step 16 and the loss is 486.8513507849441
This is step 17 and the loss is 482.2907497918997
This is step 18 and the loss is 477.7787569560063
This is step 19 and the loss is 473.31486535470435
This is

In [13]:
print(init_guess) #the parameter values recovered after applying Adam
init_guess=init_guess.detach().numpy() #turn init_guess back into array to take MSE later
print(init_guess)

tensor([[ 7.2726e-01],
        [ 7.5604e-01],
        [-2.2701e-01],
        [-2.2704e-01],
        [-2.2706e-01],
        [-2.2707e-01],
        [-2.2774e-01],
        [-2.2748e-01],
        [-2.2742e-01],
        [-2.2769e-01],
        [-2.2876e-01],
        [-2.2807e-01],
        [-2.2789e-01],
        [-2.2854e-01],
        [ 2.4354e-02],
        [ 2.3203e-02],
        [ 2.2651e-02],
        [ 2.3693e-02],
        [ 2.4223e-02],
        [ 2.2747e-02],
        [ 2.2080e-02],
        [ 2.3300e-02],
        [ 2.3618e-02],
        [ 2.2033e-02],
        [ 2.1308e-02],
        [ 2.2529e-02],
        [ 2.7579e-01],
        [ 2.7349e-01],
        [ 2.7225e-01],
        [ 2.7461e-01],
        [ 2.7761e-01],
        [ 2.7362e-01],
        [ 2.7171e-01],
        [ 2.7545e-01],
        [ 2.7888e-01],
        [ 2.7341e-01],
        [ 2.7089e-01],
        [ 2.7580e-01],
        [-2.5513e-01],
        [-2.5537e-01],
        [-2.5553e-01],
        [-2.5528e-01],
        [-6.3715e-03],
        [-6

In [14]:
#Find the MSE for the training set
pers_beta_true_array=np.array([])
pers_beta_difference_array=np.array([])
pers_delta_true_array=np.array([])
pers_delta_difference_array=np.array([])

MSE_values=np.array([])
RSE_values=np.array([])

recovered_beta_common=init_guess[0] #the recovered common component
recovered_delta_common=init_guess[1]  #the recovered personalized components for each trajectory

recovered_beta_pers=init_guess[2:len(beta_values)*len(delta_values)*num_init_cond+2] #array containing the personalized beta for each traj.
recovered_delta_pers=init_guess[len(beta_values)*len(delta_values)*num_init_cond+2:] #array containing the personalized delta for each traj.

print(recovered_beta_pers)
print(recovered_delta_pers)

count=0
for i in range(len(beta_values)): #loop through each beta_value
    for j in range(len(delta_values)): #loop through each delta_value
      for k in range(num_init_cond): #loop through each initial condition
           #you're looping thru each trajectory; there are len(beta_values)*len(delta_values)*num_init_cond total trajectories
            beta_est=recovered_beta_common[0]+recovered_beta_pers[count][0] #the estimated beta value: beta_est=beta_common_est+beta_pers_est
            delta_est=recovered_delta_common[0]+recovered_delta_pers[count][0] #the estimated delta value: delta_est=delta_common_est+delta_pers_est
            print(beta_est)
            print(delta_est)
            params=[0,alpha,0,0,-beta_values[i],0,0,-gamma,0,0,delta_values[j],0] #true parameters
            params2=[0,alpha,0,0,-beta_est,0,0,-gamma,0,0,delta_est,0] #parameters w the beta and delta values being the ones recovered from Adam

            x0=x_init_values[k] #initial condition
            y0=y_init_values[k]
            # t = np.linspace(0, 20, 100)

            results_i=odeint(SINDY_model,np.array([x0,y0]),t,args=(params,)) #true trajectory
            results_hat_i=odeint(SINDY_model,np.array([x0,y0]),t,args=(params2,)) #trajectory generated using the recovered beta and delta values

            MSE = np.square(np.subtract(results_i,results_hat_i)).mean() #MSE
            RSE=MSE/np.linalg.norm(results_i) #MSE divided by the 2-norm of the true value

            MSE_values=np.append(MSE_values,MSE)
            RSE_values=np.append(RSE_values,RSE)

            count+=1 #increment count

MSE_average=np.average(MSE_values) #find the average MSE of all trajectories
RSE_average=np.average(RSE_values) #find the average RSE for all trajectories
print("The average MSE value is " , MSE_average)
print("The average RSE value is " , RSE_average)


[[-0.22700842]
 [-0.22703946]
 [-0.22705616]
 [-0.22707098]
 [-0.22774224]
 [-0.22748252]
 [-0.22741582]
 [-0.2276939 ]
 [-0.22876438]
 [-0.22807475]
 [-0.22789312]
 [-0.22853876]
 [ 0.02435371]
 [ 0.02320318]
 [ 0.02265085]
 [ 0.02369278]
 [ 0.0242232 ]
 [ 0.02274733]
 [ 0.02208025]
 [ 0.02330024]
 [ 0.02361811]
 [ 0.02203341]
 [ 0.02130761]
 [ 0.022529  ]
 [ 0.27579081]
 [ 0.27349215]
 [ 0.27225168]
 [ 0.27461422]
 [ 0.27761215]
 [ 0.27362333]
 [ 0.27171037]
 [ 0.27544539]
 [ 0.27887696]
 [ 0.2734109 ]
 [ 0.27089163]
 [ 0.27580185]]
[[-0.25513357]
 [-0.25536779]
 [-0.2555291 ]
 [-0.25528216]
 [-0.00637152]
 [-0.00637864]
 [-0.00654222]
 [-0.00647776]
 [ 0.24131452]
 [ 0.2420466 ]
 [ 0.24200843]
 [ 0.24146968]
 [-0.25395912]
 [-0.25480285]
 [-0.25527887]
 [-0.25441154]
 [-0.00346296]
 [-0.00521896]
 [-0.00613473]
 [-0.00448506]
 [ 0.24650983]
 [ 0.24389935]
 [ 0.24253527]
 [ 0.24487395]
 [-0.25317009]
 [-0.25425276]
 [-0.25489918]
 [-0.25369542]
 [-0.00057603]
 [-0.00374985]
 [-0.0053

In [15]:
#adaptation step
# create 40 new trajectories with common component= the one we recovered during training
# find the personalized component for each trajectory and then the MSEs

beta_values_adapt=[0.625,1.125] #from the Coda paper
delta_values_adapt=[0.625,1.125]
num_init_cond_adapt=10

#x_init_values_adapt= np.random.uniform(1,3,num_init_cond_adapt) #create ten new trajectories with new values of c
#y_init_values_adapt= np.random.uniform(1,3,num_init_cond_adapt) #create ten new trajectories with new values of c

#chosen from a uniform random dist.
x_init_values_adapt=[2.80693968,2.30055217,2.57273056,2.68707454,1.54654441,2.5314465,
 1.64950653,2.20823904,1.23465885,2.35608756] #Fix the values to be same for all iterations
y_init_values_adapt=[1.41170883,2.04802652,1.67772439,1.54889085,2.73955448,2.82183897,
 1.34092985,1.47893757,1.16849701,2.58327639]

print(x_init_values_adapt)
print(y_init_values_adapt)

#make a vector of just pers components;init as 0s
init_guess_pers=np.zeros(len(init_guess_common)*len(beta_values_adapt)*len(delta_values_adapt)*num_init_cond_adapt)
init_guess_pers=torch.tensor(init_guess_pers,requires_grad=True)

[2.80693968, 2.30055217, 2.57273056, 2.68707454, 1.54654441, 2.5314465, 1.64950653, 2.20823904, 1.23465885, 2.35608756]
[1.41170883, 2.04802652, 1.67772439, 1.54889085, 2.73955448, 2.82183897, 1.34092985, 1.47893757, 1.16849701, 2.58327639]


In [16]:
#Modify the loss function to be only in terms of the personalized components; the common components are constant (set equal to the values recovered in training)
#Almost the exact same function as before; however this takes in only the pers components as variables instead of both common and pers
def loss_adaptation(init_guess_pers):
    sum=0 #initialize sum
    count=0
    # Convert recovered_beta_common and recovered_delta_common to tensors
    recovered_beta_common_tensor = torch.tensor(recovered_beta_common, dtype=torch.float64)
    recovered_delta_common_tensor = torch.tensor(recovered_delta_common, dtype=torch.float64)

    for i in range(len(beta_values_adapt)): #loop through each beta value to generate a trajectory
      for j in range(len(delta_values_adapt)): #loop through each delta value to generate a trajectory
        for k in range(num_init_cond_adapt): #loop through each initial condition
              x0=x_init_values_adapt[k]
              y0=y_init_values_adapt[k]
              params=[alpha,beta_values_adapt[i],delta_values_adapt[j],gamma] #original parameters; Fix p_l,K_A,d_k; vary c
              results=odeint(LV_model,[x0,y0],t,args=(params,))
              results= np.array(results)
              W=np.c_[results]


              x_values=W[:,0] #split into x and y vals
              y_values=W[:,1]
              fd=FiniteDifference()
              W_prime=fd._differentiate(W,t) #Take derivative


              #convert into tensors
              x_values_tensor = torch.tensor(x_values, dtype=torch.float64)
              y_values_tensor = torch.tensor(y_values, dtype=torch.float64)
              W_prime_tensor = torch.tensor(W_prime, dtype=torch.float64)


              #Estimate the derivatives
              #W_prime is the finite difference derivative
              #W_prime_hat is the estimated derivative (the right hand side of ODE, with c=init_guess_commom+init_guess_pers_i)


              W_prime=W_prime_tensor
              #exact same process as the original loss function; except the common components are fixed
            #Note: I iterate through the pers components using a count variable.
            #count is increased by 1 for each trajectory
            #in init_guess_pers, first I have all the beta_p's, then all the delta_p's
            #beta_p_(count) is at init_guess[count] #the beta_p's, start at index 0
            #delta_p_(count) is at init_guess[num_trajectories+count]
              X_hat=alpha*x_values_tensor-(recovered_beta_common_tensor+init_guess_pers[count])*x_values_tensor*y_values_tensor
              Y_hat=(recovered_delta_common_tensor+init_guess_pers[len(beta_values_adapt)*len(delta_values_adapt)*num_init_cond_adapt+count])*x_values_tensor*y_values_tensor-gamma*y_values_tensor
              W_prime_hat=torch.stack((X_hat,Y_hat),dim=1)

              Lambda_1=0.001 #Can adjust these later on
              Lambda_2=0.001 #Can adjust later on

              #Take the loss function for each trajectory and then sum them up
              sum+=torch.linalg.norm(W_prime-W_prime_hat)**2  + Lambda_2*(torch.linalg.norm(torch.tensor([init_guess_pers[count],init_guess_pers[len(beta_values_adapt)*len(delta_values_adapt)*num_init_cond_adapt+count]], dtype=torch.float64)))**2
              count+=1
              #+ Lambda_1*torch.linalg.norm(torch.tensor([recovered_beta_common_tensor+init_guess_pers[count],recovered_delta_common_tensor+init_guess_pers[len(beta_values_adapt)*len(delta_values_adapt)*num_init_cond_adapt+count]], dtype=torch.float64), ord=1)
    return sum

In [17]:
print(loss_adaptation(init_guess_pers))

tensor(687.2486, dtype=torch.float64, grad_fn=<AddBackward0>)


In [18]:
#Use Adam to personalize each trajectory in the Adaptation

init_guess_pers=torch.tensor(init_guess_pers, requires_grad=True)
optimizer=optim.Adam([init_guess_pers],lr=0.001) #creates an Adam Optimizer object, lr is learning rate, use the default value from PyTorch
prev=0 #represent the loss value of the previous iteration
step=0 #the iteration number
loss_value=100 #use to represent the current loss value; I initialized as a random value just so loss_value and prev wouldn't start out the same
while(abs(loss_value-prev)>0.0001): #until convergence (algorithm terminates when current and previous loss function are within 0.0001 of each other)
  prev=loss_value #set prev=the loss value of the previous iteration
  optimizer.zero_grad() #reset the gradient
  loss_value=loss_adaptation(init_guess_pers) #take loss value of current iteration
  loss_value.backward() #take the gradient
  optimizer.step() #update parameters using Adam, step forward
  print("This is step", step, "and the loss is", loss_value.item())
  step+=1

This is step 0 and the loss is 687.2485965764357
This is step 1 and the loss is 682.7267047589444
This is step 2 and the loss is 678.2254227712456
This is step 3 and the loss is 673.7450972805539
This is step 4 and the loss is 669.286067146197
This is step 5 and the loss is 664.8486625562797
This is step 6 and the loss is 660.4332042049502
This is step 7 and the loss is 656.0400025162066
This is step 8 and the loss is 651.6693569195738
This is step 9 and the loss is 647.3215551823231
This is step 10 and the loss is 642.9968728021747
This is step 11 and the loss is 638.6955724636754
This is step 12 and the loss is 634.4179035606559
This is step 13 and the loss is 630.1641017863765
This is step 14 and the loss is 625.9343887922132
This is step 15 and the loss is 621.7289719149479
This is step 16 and the loss is 617.5480439720337
This is step 17 and the loss is 613.3917831235088
This is step 18 and the loss is 609.2603527986395
This is step 19 and the loss is 605.1539016847881
This is ste

In [19]:
init_guess_pers=init_guess_pers.detach().numpy() #turn init_guess back into array
print(init_guess_pers)
print(init_guess_pers.shape)

[-0.1033529  -0.10262191 -0.10308262 -0.10322611 -0.10164698 -0.10181022
 -0.10254936 -0.10287464 -0.10236028 -0.10206555 -0.10586754 -0.10434658
 -0.10528    -0.10558765 -0.10205713 -0.10323029 -0.10349497 -0.1045174
 -0.10289666 -0.10351114  0.39600556  0.39919397  0.39750151  0.39678574
  0.39881113  0.40204699  0.396746    0.39683877  0.39658718  0.40106045
  0.39226549  0.39949967  0.39541852  0.39387038  0.40258374  0.40746233
  0.39548072  0.39480166  0.39576341  0.4046739  -0.13170639 -0.13080479
 -0.13134748 -0.13153058 -0.12990819 -0.12983954 -0.13105046 -0.13125711
 -0.13099118 -0.13014575  0.36293143  0.36606918  0.36419125  0.36354671
  0.37002998  0.36855618  0.36680594  0.3652566   0.36762102  0.36787881
 -0.13038005 -0.12871509 -0.12953714 -0.12992881 -0.12963955 -0.12751708
 -0.13019919 -0.13000531 -0.13042554 -0.12794185  0.36626641  0.37348941
  0.36953175  0.36795404  0.37527626  0.38102631  0.36890719  0.36859989
  0.36885998  0.37835612]
(80,)


In [20]:
#Find the MSE of adaptation
pers_beta_array=[]
pers_delta_array=[]
MSE_values=[]
RSE_values=[]
pers_c_difference_array=[]
count=0

for i in range(len(beta_values_adapt)): #loop through each beta_value in adaptation
    for j in range(len(delta_values_adapt)): #loop through each delta_value in adaptation
      for k in range(num_init_cond_adapt): #loop through each initial condition
            x0=x_init_values_adapt[k]
            y0=y_init_values_adapt[k]

            params=[0,alpha,0,0,-beta_values_adapt[i],0,0,-gamma,0,0,delta_values_adapt[j],0] #true parameters of the Lotka_Volterra
            results=odeint(SINDY_model,np.array([x0,y0]),t,args=(params,)) #generate the true trajectories

            #separate into x and y values
            x_values_adapt=results[:,0] #x_values of traj
            y_values_adapt=results[:,1] #y-values of traj

            beta_est=recovered_beta_common[0]+init_guess_pers[count] #our estimated beta; equal to beta_c_i (from training) + beta_p (from adaptation)
            delta_est=recovered_delta_common[0]+init_guess_pers[len(beta_values_adapt)*len(delta_values_adapt)*num_init_cond_adapt+count]
            pars=[0,alpha,0,0,-beta_est,0,0,-gamma,0,0,delta_est,0]
            results_hat=odeint(SINDY_model,np.array([x0,y0]),t,args=(pars,)) #generate new traj using the estimated betas and deltas
            print("Betas", beta_values_adapt[i],beta_est)

            print("Deltas", delta_values_adapt[j],delta_est)


            MSE = np.square(np.subtract(results,results_hat)).mean()
            RSE=MSE/np.linalg.norm(results)
            print("The MSE for trajectory ", count, "is", MSE)
            print("The RSE for trajectory ", count, "is", RSE)
            print()


            MSE_values=np.append(MSE_values,MSE)
            RSE_values=np.append(RSE_values,RSE)

            pers_beta_array=np.append(pers_beta_array,init_guess_pers[count])
            pers_delta_array=np.append(pers_delta_array,init_guess_pers[len(beta_values_adapt)*len(delta_values_adapt)*num_init_cond_adapt+count])
            count+=1

#personalized components
print("Est personalized beta", pers_beta_array)
print("Est personalized delta", pers_delta_array)

#the MSE and RSE for adaptation
MSE_average=np.average(MSE_values)
RSE_average=np.average(RSE_values)
print("The average MSE value is " , MSE_average)
print("The average RSE value is " , RSE_average)

Betas 0.625 0.6239054296780989
Deltas 0.625 0.6243341770487036
The MSE for trajectory  0 is 4.067660043600153e-06
The RSE for trajectory  0 is 1.0034500648047147e-07

Betas 0.625 0.6246364257474312
Deltas 0.625 0.625235776381022
The MSE for trajectory  1 is 3.99832834553194e-06
The RSE for trajectory  1 is 9.458515141868084e-08

Betas 0.625 0.6241757166585936
Deltas 0.625 0.6246930924388484
The MSE for trajectory  2 is 3.902389632559758e-06
The RSE for trajectory  2 is 9.503495353994525e-08

Betas 0.625 0.6240322278379518
Deltas 0.625 0.6245099932305108
The MSE for trajectory  3 is 3.917177013910821e-06
The RSE for trajectory  3 is 9.595664505815896e-08

Betas 0.625 0.6256113493853442
Deltas 0.625 0.6261323818048466
The MSE for trajectory  4 is 1.563954937857442e-06
The RSE for trajectory  4 is 3.718503871673429e-08

Betas 0.625 0.6254481116469837
Deltas 0.625 0.6262010283058258
The MSE for trajectory  5 is 7.490716051794203e-06
The RSE for trajectory  5 is 1.4263927105640023e-07

Beta

In [21]:
# #initialize empty arrays
# pers_beta_true_array=np.array([])
# pers_beta_difference_array=np.array([])
# pers_delta_true_array=np.array([])
# pers_delta_difference_array=np.array([])

# MSE_values=np.array([])
# RSE_values=np.array([])

# #recovered_beta_common=init_guess[0] #the recovered common component
# #recovered_delta_common=init_guess[1]  #the recovered personalized components for each trajectory

# recovered_beta_pers=init_guess_pers[0:len(beta_values_adapt)*len(delta_values_adapt)*num_init_cond_adapt] #array containing the personalized beta for each traj.
# recovered_delta_pers=init_guess_pers[len(beta_values_adapt)*len(delta_values_adapt)*num_init_cond_adapt:] #array containing the personalized delta for each traj.

# print(recovered_beta_pers)
# print(recovered_delta_pers)

# #print(recovered_common)
# #print(recovered_pers)
# #print(recovered_pers.shape)
# count=0
# for i in range(len(beta_values_adapt)): #loop through each beta_value
#     for j in range(len(delta_values_adapt)): #loop through each delta_value
#       for k in range(num_init_cond_adapt): #loop through each initial condition
#            #you're looping thru each trajectory; there are len(beta_values)*len(delta_values)*num_init_cond total trajectories
#             beta_est=recovered_beta_common[0]+recovered_beta_pers[count] #the estimated beta value: beta_est=beta_common_est+beta_pers_est
#             delta_est=recovered_delta_common[0]+recovered_delta_pers[count] #the estimated delta value: delta_est=delta_common_est+delta_pers_est
#             print(beta_est)
#             print(delta_est)
#             params=[0,alpha,0,0,-beta_values_adapt[i],0,0,-gamma,0,0,delta_values_adapt[j],0] #true parameters
#             params2=[0,alpha,0,0,-beta_est,0,0,-gamma,0,0,delta_est,0] #parameters w recovered c_value
#             #print("sum is", i+j+k)

#             x0=x_init_values_adapt[k] #initial condition
#             y0=y_init_values_adapt[k]
#             # t = np.linspace(0, 20, 100)
#             #results_i = odeint(logistic_growth_controlled, La_o , t, args=(params, treatment_function, cycles)) #trajectory generated using the true c values
#             #results_hat_i=odeint(logistic_growth_controlled,La_o,t,args=(params2,treatment_function,cycles))    #trajectory generated using the recovered c values

#             results_i=odeint(SINDY_model,np.array([x0,y0]),t,args=(params,)) #true trajectory
#             results_hat_i=odeint(SINDY_model,np.array([x0,y0]),t,args=(params2,)) #trajectory with estimated value for c

#             plt.plot(t, results_i[:,0], label='x')
#             plt.plot(t, results_i[:,1], label='y')
#             plt.plot(t, results_hat_i[:,0], label='x_hat')
#             plt.plot(t, results_hat_i[:,1], label='Y_hat')
#             plt.yscale('linear')
#             plt.legend()
#             plt.show()

#             MSE = np.square(np.subtract(results_i,results_hat_i)).mean() #MSE
#             RSE=MSE/np.linalg.norm(results_i) #MSE divided by the 2-norm of the true value
#             #print("The MSE is ", MSE)
#             #print("The RSE is ", RSE)

#             MSE_values=np.append(MSE_values,MSE)
#             RSE_values=np.append(RSE_values,RSE)

#             count+=1


#             #pers_c_true=c_values[i]-true_c_common #the "true" personalized component c_(p_i) (c_p_i=c_i-c_c)
#             #pers_c_true_array=np.append(pers_c_true_array,pers_c_true) #add to an array

#             #pers_c_true_difference=pers_c_true-recovered_pers[i] #the difference between the true and recovered personalized components
#             #pers_c_difference_array=np.append(pers_c_difference_array,pers_c_true_difference)

# #print("The true common component is ", true_c_common)
# #print("The recovered common component is ", recovered_common)
# #print("Difference in common components", true_c_common-recovered_common)

# #print("True personalized comp", pers_c_true_array)
# #print("Est personalized comp", recovered_pers)
# #print("Difference in personalized components", pers_c_difference_array)



# MSE_average=np.average(MSE_values) #find the average MSE of all trajectories
# RSE_average=np.average(RSE_values) #find the average RSE for all trajectories
# print("The average MSE value is " , MSE_average)
# print("The average RSE value is " , RSE_average)